# VAME on a full reference dataset

This tutorial runs the full VAME workflow, step by step, on a complete reference dataset read from [DANDI](https://dandiarchive.org/): the Palop Lab J20-FVB open-field dataset. It holds open-field recordings of hAPP-J20 transgenic mice and wild-type mice on an FVB background, 109 sessions from 55 mice, with poses tracked by DeepLabCut (11 keypoints) and stored as NWB files with `ndx-pose`.

The steps are the same for any dandiset with `ndx-pose` data, simply change the settings in the next section to use another one.

Training on all sessions is much faster on a GPU. VAME uses one when available (CUDA or Apple MPS) and falls back to the CPU otherwise.

In [ ]:
%pip install "vame-py[dandi]>=0.15.0" python-dotenv

After installing, restart the kernel before running the next cells.

## Settings

The dandiset to read and where to create the project. The parameters of each step are set in that step's cell.

In [ ]:
from pathlib import Path

DANDI_INSTANCE = "dandi-sandbox"
DANDISET_ID = "219084"
VERSION = "draft"

WORKING_DIR = Path.home() / "vame-projects"
PROJECT_NAME = "j20"

## DANDI API key

The DANDI client reads the key from an environment variable named after the instance: `DANDI_SANDBOX_API_KEY` for the sandbox, `DANDI_API_KEY` for the main archive. Get your key from your account page on the DANDI website.

Set it in one of two ways:

- Put it in a `.env` file in the folder you run the notebook from, with the line `DANDI_SANDBOX_API_KEY=<your api key>`. `load_dotenv` reads it into the environment. Keep `.env` out of version control.
- Set it in the notebook with `os.environ`, replacing the placeholder. Don't commit or share the notebook with a real key in it.

In [ ]:
import os
from dotenv import load_dotenv

# Option 1: read the key from a .env file
load_dotenv()

# Option 2: set the key here
# os.environ["DANDI_SANDBOX_API_KEY"] = "<your api key>"

Check which device VAME will train on: a CUDA GPU, an Apple silicon GPU through MPS, or the CPU.

In [ ]:
import torch

# Same order as VAME: CUDA, then Apple MPS, then CPU
if torch.cuda.is_available():
    print("Device: CUDA GPU,", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print("Device: Apple MPS GPU")
else:
    print("Device: CPU (no GPU found)")

## 1. Find the pose data in the dandiset

`dandiset_parse` opens each NWB file and reports which ones have pose estimation data, and how many files have each keypoint.

In [ ]:
from vame.io.dandi import dandiset_parse, dandiset_load

parsed = dandiset_parse(DANDISET_ID, VERSION, dandi_instance=DANDI_INSTANCE)

n_files = len(parsed["valid_files"])
print(f"{n_files} NWB files with pose data")
parsed["pose_estimation_series"]

## 2. Load the poses

VAME needs the same keypoints in every session, so keep the keypoints present in all files. `dandiset_load` writes one movement `.nc` file per session. With `download=True`, each NWB file is downloaded to a temporary folder, read, and deleted, which is faster than streaming for files that hold little besides pose data.

In [ ]:
keypoints = [s["name"] for s in parsed["pose_estimation_series"] if s["count"] == n_files]
print("Keypoints:", keypoints)

poses_estimations = dandiset_load(
    DANDISET_ID,
    VERSION,
    files=parsed["valid_files"],
    pose_estimation_series=keypoints,
    out_dir=str(WORKING_DIR / "dandi_nc"),
    dandi_instance=DANDI_INSTANCE,
    download=True,
)
print(f"{len(poses_estimations)} sessions loaded")

## 3. Create the VAME project

Each session is named after its NWB file's path in the dandiset, for example `sub-1_sub-1_ses-MetaSession21_image`. Project parameters are stored in the project's `config.yaml`. This notebook sets the ones each step reads right before that step, with `update_config`, which also saves them to `config.yaml`. See the [project configuration](https://ethoml.github.io/VAME/docs/project-config/) page for all of them. If the project already exists, it is reused.

In [ ]:
import vame

config_file, config_data = vame.init_new_project(
    project_name=PROJECT_NAME,
    poses_estimations=poses_estimations,
    source_software="movement",
    working_directory=str(WORKING_DIR),
)
print(config_file)

## 4. Preprocess

Cleans low-confidence points, aligns each pose on the two reference keypoints, removes outliers, smooths with a Savitzky-Golay filter, and rescales each animal to its body size.

`pose_confidence` is the confidence below which points are cleared and interpolated. Poses are aligned so that the tail base is at the center and the nose points in a fixed direction.

In [ ]:
config_data = vame.update_config(config=config_data, config_update={"pose_confidence": 0.9})

vame.preprocessing(
    config=config_data,
    centered_reference_keypoint="Tailbase",
    orientation_reference_keypoint="Nose",
)

Plot one session's keypoint positions after each preprocessing step: raw, aligned, filtered and rescaled. By default it shows the first 1000 frames of the first session.

In [ ]:
from vame.visualization import visualize_preprocessing_timeseries

visualize_preprocessing_timeseries(config=config_data)

## 5. Train and evaluate the model

`create_trainset` builds the training and test sets from the preprocessed poses and saves the normalization statistics the model is trained with. The keypoints in `keypoints_to_exclude` stay in the project, but are left out of the model input.

In [ ]:
vame.create_trainset(config=config_data, keypoints_to_exclude=["Midtail", "Tailtip"])

`train_model` trains the recurrent variational autoencoder. Here it trains for at most 100 epochs of 200 batches each, and stops early when the test loss has not improved for `model_convergence` epochs. The model's input window (`time_window`) and latent size (`zdims`) keep their defaults. Evaluation, segmentation and the videos read them too, so changing them after training means training again.

In [ ]:
config_data = vame.update_config(
    config=config_data,
    config_update={
        "max_epochs": 100,
        "steps_per_epoch": 200,
    },
)

vame.train_model(config=config_data)

`evaluate_model` plots reconstructions and future predictions of test samples to `model/evaluate`, and `plot_loss` plots the training and test losses over epochs. The last line shows the saved future reconstruction plot.

In [ ]:
from IPython.display import Image, display
from vame.visualization import plot_loss

vame.evaluate_model(config=config_data)
plot_loss(config=config_data)
display(Image(filename=Path(config_data["project_path"]) / "model" / "evaluate" / "future_reconstruction.png"))

## 6. Segment behavior into motifs and communities

`segment_session` encodes each session into the model's latent space and assigns every frame to one of `n_clusters` motifs (15 here). The latent vectors are saved in each session's results folder and reused by later runs. To redo a segmentation that already exists, pass `overwrite_segmentation=True`.

This run uses k-means only, with one initialization (`kmeans_n_init`) fit on a random sample of 1,000,000 frames (`kmeans_fit_samples`), and every frame is still assigned a motif. On a dataset this size, that is much faster than the defaults (20 initializations on all frames).

HMM segmentation (`"hmm"` and `"hmm_warmstart"`) is also available.

In [ ]:
config_data = vame.update_config(
    config=config_data,
    config_update={
        "n_clusters": 15,
        "segmentation_algorithms": ["kmeans"],
        "kmeans_n_init": 1,
        "kmeans_fit_samples": 1_000_000,
    },
)

vame.segment_session(config=config_data)

Motif usage per session, in percent of frames and sorted from most to least used, with the mean across sessions in red. Motifs below the 1% threshold line are rarely used.

In [ ]:
from vame.visualization import visualize_motif_thresholding

visualize_motif_thresholding(config=config_data, segmentation_algorithm="kmeans")

`community` groups motifs into communities with a hierarchical tree built from the transitions between motifs. `cut_tree` sets the level at which the tree is cut, and with it the number of communities.

In [ ]:
vame.community(config=config_data, cut_tree=2)

The hierarchical tree of motifs that the communities come from.

In [ ]:
from vame.visualization import visualize_hierarchical_tree

visualize_hierarchical_tree(config=config_data, segmentation_algorithm="kmeans")

`visualize_umap` embeds 30,000 latent vectors in two dimensions with UMAP and plots them, also colored by motif and by community. The figures are saved in `reports/umap`.

In [ ]:
from vame.visualization import visualize_umap

visualize_umap(config=config_data, show_figure="matplotlib")

## 7. Motif and community videos for one session

Motif and community videos are short clips of the original recording, one per motif or community, and need the session's video. The J20 videos are about 0.5 GB each, so this section downloads the video of a single session and makes clips for it only. In the dandiset, each session's video is in a folder next to its NWB file, with the same name. VAME looks for it in the project's `data/raw` folder, named after the session.

In [ ]:
from dandi.dandiapi import DandiAPIClient

VIDEO_SESSION = config_data["session_names"][0]

session_to_nwb = {path.replace("/", "_").removesuffix(".nwb"): path for path in parsed["valid_files"]}
video_folder = session_to_nwb[VIDEO_SESSION].removesuffix(".nwb") + "/"

with DandiAPIClient.for_dandi_instance(DANDI_INSTANCE) as client:
    client.dandi_authenticate()
    dandiset = client.get_dandiset(DANDISET_ID, VERSION)
    video_asset = next(a for a in dandiset.get_assets_with_path_prefix(video_folder) if a.path.endswith(".avi"))
    video_path = Path(config_data["project_path"]) / "data" / "raw" / f"{VIDEO_SESSION}.avi"
    video_asset.download(video_path)

print(video_path)

Write the clips for this session only. `motif_videos` writes one clip per motif and `community_videos` one per community, each up to `length_of_motif_video` frames (1000 by default).

In [ ]:
vame.motif_videos(config=config_data, sessions=[VIDEO_SESSION])
vame.community_videos(config=config_data, sessions=[VIDEO_SESSION])

Show the first motif clip. The clips are saved in the session's results folder, in `cluster_videos` for motifs and `community_videos` for communities.

In [ ]:
from IPython.display import Video

results_path = (
    Path(config_data["project_path"]) / "results" / VIDEO_SESSION / config_data["model_name"] / f"kmeans-{config_data['n_clusters']}"
)
motif_clips = sorted((results_path / "cluster_videos").glob("*.mp4"))
print(f"{len(motif_clips)} motif clips in {results_path / 'cluster_videos'}")
Video(motif_clips[0], embed=True, width=400)